# Monte Carlo Recovery (two-type CCP–EM with fixes)

In [1]:
# %pip -q install numpy pandas scipy matplotlib

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from simulate_data import SimConfig, simulate_panel
from estimate_npl_em import estimate_npl_em, load_data


In [3]:
# CSV = "./data/simulated_panel.csv"  # provided dataset
# OUT = "./data/mc_ccp_em"
# os.makedirs(OUT, exist_ok=True)
#
# df = load_data(CSV)
# df.head()

,id,t,a,y,x,age
0,1,0,51882.159995,25000.0,0.00,30.0
1,1,1,62335.842556,25000.0,0.25,31.0
2,1,2,56719.175390,25000.0,1.00,32.0
3,1,3,47818.190334,25000.0,0.25,33.0
4,1,4,34817.556638,25000.0,0.25,34.0


In [ ]:
base_const = dict(beta=0.93, sigma=2.2, R_f=1.02, mu_lnR=np.log(1.06), sigma_lnR=0.20)
theta1_true = (1.2, -0.008, -0.8, 0.012)
theta2_true = (2.5,  0.015, -1.8, 0.020)
eta_true    = np.array([-0.5, -0.02, 0.45])
cfg = SimConfig(N=400, T=8, NA_grid=24, gh_order=5, a_min=5000, a_max=80000, age_start=30, y_bar=25000.0)

MC = 1
OUTROOT = "./data/mc_ccp_em"
os.makedirs(OUTROOT, exist_ok=True)
rows = []
for m in range(MC):
    df = simulate_panel(cfg, base_const, theta1_true, theta2_true, eta_true, rng_seed=100+m)
    csv = os.path.join(OUTROOT, f"sim_{m}.csv"); df.to_csv(csv, index=False)
    outdir = os.path.join(OUTROOT, f"est_{m}"); os.makedirs(outdir, exist_ok=True)
    est = estimate_npl_em(csv, outdir=outdir, restarts=6, seed=m,
            n_a=24, gh_order=5, em_iters=6, nm_steps=25,
            fix_beta_sigma=True, beta_fix=base_const['beta'], sigma_fix=base_const['sigma'],
            damping=0.5)
    rows.append(dict(
        m=m,
        g0_1=est["theta1"][0], g1_1=est["theta1"][1], d0_1=est["theta1"][2], d1_1=est["theta1"][3],
        g0_2=est["theta2"][0], g1_2=est["theta2"][1], d0_2=est["theta2"][2], d1_2=est["theta2"][3],
        eta0=est["eta"][0], eta1=est["eta"][1], eta2=est["eta"][2],
        obs_ll=est["obs_ll"]
    ))
res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUTROOT, "mc_results.csv"), index=False)
res.head()


In [ ]:

true = pd.Series({
    'g0_1': theta1_true[0], 'g1_1': theta1_true[1], 'd0_1': theta1_true[2], 'd1_1': theta1_true[3],
    'g0_2': theta2_true[0], 'g1_2': theta2_true[1], 'd0_2': theta2_true[2], 'd1_2': theta2_true[3],
    'eta0': eta_true[0], 'eta1': eta_true[1], 'eta2': eta_true[2]
})
means = res.drop(columns=['m','obs_ll']).mean()
bias = means - true
rmse = ((res.drop(columns=['m','obs_ll']) - true)**2).mean()**0.5
pd.DataFrame({'true': true, 'mean': means, 'bias': bias, 'rmse': rmse})
